[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/10_Graph_Analytics.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 图分析（Graph Analytics）

## 概览

本 notebook 演示如何用 Semantica 的分析模块分析知识图谱。你将学会使用 `GraphAnalyzer`、`CentralityCalculator`、`CommunityDetector` 和 `ConnectivityAnalyzer` 来理解图的结构与性质。

**文档**：[API 参考](https://semantica.readthedocs.io/reference/kg/)

### 先看主线

图建好了——07 篇组装、08 篇首航、09 篇入库。这一篇开始"读图"：拿四个问题由浅入深地盘问一张图，四步各答一问：

1. **整体长什么样？**——图分析（第 1 步）：实体数、关系数、密度等指标体检
2. **哪些节点最重要？**——中心性（第 2 步）：找出图里的关键角色
3. **节点怎么抱团？**——社区发现（第 3 步）：找出图里的圈子
4. **图有没有断开？**——连通性（第 4 步）：查整张图的骨架

四问从整体到个体、从个体到群体，最后回到整体骨架。答案也不是看完就完——高中心性节点适合做检索入口，社区天然是聚类视图，连通分量告诉你图缺了哪些桥。

### 学习目标

- 使用 `GraphAnalyzer` 做全面的图分析
- 使用 `CentralityCalculator` 计算中心性度量
- 使用 `CommunityDetector` 发现图中的社区
- 使用 `ConnectivityAnalyzer` 分析图的连通性

## 安装

从 PyPI 安装 Semantica：

```bash
pip install semantica
# 或安装全部可选依赖：
pip install semantica[all]
```

---

## 第 1 步：图分析

第一问：整体长什么样？分析图的结构和性质。

In [ ]:
!pip install semantica

In [ ]:
from semantica.kg import GraphBuilder, GraphAnalyzer
from semantica.semantic_extract import NERExtractor, RelationExtractor

builder = GraphBuilder()
analyzer = GraphAnalyzer()

entities = [
    {"id": "e1", "type": "Organization", "name": "Apple Inc.", "properties": {}},
    {"id": "e2", "type": "Person", "name": "Tim Cook", "properties": {}},
    {"id": "e3", "type": "Location", "name": "Cupertino", "properties": {}}
]

relationships = [
    {"source": "e2", "target": "e1", "type": "CEO_of", "properties": {}},
    {"source": "e1", "target": "e3", "type": "located_in", "properties": {}}
]

kg = builder.build(entities, relationships)

metrics = analyzer.compute_metrics(kg)

print(f"Graph metrics:")
print(f"  Entities: {metrics.get('entity_count', 0)}")
print(f"  Relationships: {metrics.get('relationship_count', 0)}")
print(f"  Density: {metrics.get('density', 0):.3f}")


## 第 2 步：中心性度量

第二问：哪些节点最重要？给实体计算中心性分数——分数越高，节点在图里越关键。

In [ ]:
from semantica.kg import CentralityCalculator

centrality_calculator = CentralityCalculator()

centrality_result = centrality_calculator.calculate_degree_centrality(kg)
centrality_scores = centrality_result.get('centrality', {})

print(f"Centrality scores:")
for entity_id, score in list(centrality_scores.items())[:5]:
    print(f"  {entity_id}: {score:.3f}")


## 第 3 步：社区发现

第三问：节点怎么抱团？在图中检测社区——联系紧密的节点天然聚成圈子。

In [ ]:
from semantica.kg import CommunityDetector

community_detector = CommunityDetector()

# Get detection result
result = community_detector.detect_communities(kg)

# Extract communities list from result dictionary
communities = result.get("communities", [])

print(f"Detected {len(communities)} communities")
for i, community in enumerate(communities[:3], 1):
    print(f"  Community {i}: {len(community)} entities")

## 第 4 步：连通性分析

第四问：图有没有断开？分析图的连通性——是一整块，还是散成几个孤岛。

In [ ]:
from semantica.kg import ConnectivityAnalyzer

connectivity_analyzer = ConnectivityAnalyzer()

connectivity = connectivity_analyzer.analyze_connectivity(kg)

print(f"Connectivity analysis:")
print(f"  Is connected: {connectivity.get('is_connected', False)}")
print(f"  Components: {len(connectivity.get('components', []))}")


## 小结

### 回顾主线

读图四问，各有所答：指标体检看整体、中心性找关键、社区发现看抱团、连通性查骨架。读图之后，下一件自然的事是数据质量——图里的重复实体怎么归一，正是后续去重（Deduplication）篇的主题。

### 你学到了什么

你已学会如何分析知识图谱：

- **GraphAnalyzer**：全面的图分析与指标
- **CentralityCalculator**：计算中心性度量
- **CommunityDetector**：发现图中的社区
- **ConnectivityAnalyzer**：分析图的连通性

下一步：在 Deduplication notebook 中学习如何给实体去重。